In [263]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


In [265]:
query_com = """
SELECT DISTINCT v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-02-20'
    AND v.Fecha <= '2026-02-22'
    AND v.Venta_Neta > 0
    AND Categoria IN ('CUIDADO FACIAL')
GROUP BY  v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas 

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,4354270754,C41499174,ROSA MARGARITA GIRALDO HURTADO,rosamargaritagiraldo@gmail.com,3103073395,2026-02-20,Chat Center,1114285.72
1,4354270824,C41494420,MARIANA WILD,mwild@transwill.com,3118103861,2026-02-20,Retail,619159.67
2,4354270843,C37834794,MARIA CUSTODIA BUENAHORA LOZA,negado@provenzal.net,3002185284,2026-02-20,Retail,83193.28
3,4354270865,C52799041,LUZ DARY HERNANDEZ GUAYAMBUCO,ldhernandezg@gmail.com,3208702446,2026-02-20,Chat Center,326050.42
4,4354270867,C41539113,LINA MARIA AMEZQUITA MATEUS,negado@provenzal.net,3176810080,2026-02-20,Retail,104201.68
...,...,...,...,...,...,...,...,...
166,96586,C32683039,LUCIA ISABEL RUIZ MARTINEZ,luciaruiz077@gmail.com,3103620893,2026-02-21,Shopify,584218.57
167,96587,C1010241805,BIBIANA SORA,bibianasora@gmail.com,3213964522,2026-02-21,Shopify,523086.75
168,96588,C1010241805,BIBIANA SORA,bibianasora@gmail.com,3213964522,2026-02-21,Shopify,393932.64
169,96591,C51654357,OLGA CECILIA PEREZ RODRIGUEZ,olga.perez.r@gmail.com,3166947726,2026-02-22,Shopify,908784.46


In [266]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT v.Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2026-01-03'
#     AND v.Fecha <= '2026-01-31'
#     AND v.Venta_Neta > 0
# AND Sku IN ('38UV030A25','29HD250A15','29EC200A26','29GE250A26','29LC200A22')
# GROUP BY  v.Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [267]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'CUMP_FEB2026')
# 	AND Fecha BETWEEN '2026-02-14' AND '2026-02-28'
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [268]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clicks'
}, inplace=True)

# 1. Convertir la columna openedAt de string a lista real de fechas
df_campania['apertura'] = df_campania['apertura'].astype(str)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: ast.literal_eval(x) if x.startswith("[") else []
)

# 2. Expandir: cada fecha en una fila
df_expanded = df_campania.explode('apertura').copy()

# 3. Limpiar y convertir a datetime
df_expanded['apertura'] = df_expanded['apertura'].str.strip()
df_expanded['apertura'] = pd.to_datetime(
    df_expanded['apertura'],
    format='%Y-%m-%dT%H:%M:%S.%fZ',
    utc=True,
    errors='coerce'
).dt.date

#Convertir la columna clicks en Fecha
df_campania['clicks'] = df_campania['clicks'].astype(str)

def extraer_fecha_click(valor):
    try:
        # Convertir string a objeto Python (lista de dicts)
        lista = ast.literal_eval(valor)
        if isinstance(lista, list) and len(lista) > 0 and 'clicked_at' in lista[0]:
            fecha = pd.to_datetime(lista[0]['clicked_at'], utc=True, errors='coerce')
            return fecha.date()  # solo fecha
    except Exception:
        return pd.NaT
    return pd.NaT

df_campania['clicks'] = df_campania['clicks'].apply(extraer_fecha_click)

# 4. Normalizar correos
df_expanded['Email'] = df_expanded['Email'].str.lower().str.strip()

# 5. Elegir solo la primera apertura por email
df_abiertos = df_expanded.dropna(subset=['apertura']) \
    .sort_values('apertura') \
    .drop_duplicates(subset='Email', keep='first')

df_clics =  df_campania[df_campania['clicks'].notnull()].copy()

# Filtro correos abiertos hasta una fecha maxima
fecha_maxima = pd.to_datetime('2026-02-22').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'] <= fecha_maxima]
df_clicks_filtrados = df_clics[df_clics['clicks'] <= fecha_maxima]
total_abiertos = df_abiertos_filtrados['Email'].nunique()
total_clicks = df_clicks_filtrados['Email'].nunique()

In [269]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_abiertos, on='Email', how='inner')

# Filtrar ventas dentro de 5 días desde la apertura
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['apertura']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")
print(f"Número de clientes que hicieron clics hasta {fecha_maxima}: {total_clicks}")
print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Número de clientes que abrieron hasta 2026-02-22: 3429
Número de clientes que hicieron clics hasta 2026-02-22: 49
Total de Ventas: 9
Total de venta atribuida: 3,203,652.28


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,apertura,clicks,dias_post_apertura
1,4354271286,C1015437321,AIDA MANTILLA,aidamantilla@hotmail.com,3132089423,2026-02-20,Retail,148739.50,ÃšLTIMOS DÃAS ESCALONADO FACIAL,2026-02-20T17:16:05.000Z,2026-02-20,[],0
3,4354272053,C24340615,AURA SOFIA GIL VERA,sofiagilv@hotmail.com,3234775616,2026-02-21,Retail,66386.55,ÃšLTIMOS DÃAS ESCALONADO FACIAL,2026-02-20T17:16:43.000Z,2026-02-20,[],1
4,4354272107,C43875806,MONICA MARIA GIRALDO RESTREPO,mogiraldo@yahoo.com,3182972391,2026-02-21,Retail,421848.74,ÃšLTIMOS DÃAS ESCALONADO FACIAL,NaN,2026-02-21,[],0
5,4354272306,C8029543,SEBASTIAN LOPEZ,sebaslo96@hotmail.com,3024368251,2026-02-21,Retail,110924.37,ÃšLTIMOS DÃAS ESCALONADO FACIAL,2026-02-20T17:17:01.000Z,2026-02-20,[],1
6,4354272335,C1026259060,LUIS CARLOS ESPITIA MELO,luiscarlosespitia13@gmail.com,3174270413,2026-02-21,Retail,371218.48,ÃšLTIMOS DÃAS ESCALONADO FACIAL,2026-02-20T17:16:09.000Z,2026-02-20,[],1


In [270]:
df_atribucion.to_excel('Ventas.xlsx', index=False)